# Pairs Trading

Pairs trading bets that the **gap** between two assets goes back to normal, whatever the market does. The idea is simple. The hard part is proving that the gap really closes and is not just an illusion of the chart. This notebook builds the whole chain on ETF data: the difference between correlation and cointegration, the ADF test, the hedge ratio, Engle-Granger (with the right critical values), Johansen, half-life and Hurst, a dynamic hedge ratio (rolling OLS and ALMA filter), screening of a whole universe with multiple testing correction, signals, a backtest with costs, and the in-sample / out-of-sample comparison. The backtests are demonstrations (daily bars, ETFs, one period), not validations.

**Contents**

1. Data and the pair, by eye
2. Spurious correlation: prices vs returns, correlation vs cointegration
3. Stationarity and the ADF test
4. Hedge ratio and spread: OLS in both directions, Total Least Squares
5. Engle-Granger and the CADF test: mind the critical values
6. The Johansen test on a 3-asset basket
7. Universe screening, p-value heatmap and the multiple testing problem
8. Half-life and Hurst exponent
9. Dynamic hedge ratio: static vs rolling OLS vs ALMA
10. Z-score and trading signals
11. Backtest with costs and the look-ahead trap
12. Results: in-sample vs out-of-sample, and the relationship breakdown

## 0. Setup and data

Install the dependencies with `pip install numpy pandas scipy statsmodels yfinance plotly joblib`.

The parameters below are the only knobs of the notebook. Everything that is *estimated* (hedge ratio, screening, half-life, z-score window) is estimated on the **in-sample** period only (up to `IS_END`); the later years are used only to see what survives. The universe is made of country and sector ETFs, so that every pair has at least a plausible economic link.

In [22]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from joblib import Parallel, delayed
from itertools import combinations
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_dark"
# if charts don't show up (some VS Code / JupyterLab versions): pio.renderers.default = "notebook"

In [23]:
# --- data and sample split ---
UNIVERSE = ("EWA EWC EWZ EWY EWT EWH EWS EWG EWU EWJ EWQ EWI EWP EWL EWD EWN EWW EWM EZA "
            "SPY QQQ XLE XLF XLK XLV XLY XLP XLI XLU XLB GLD SLV GDX USO IWM DIA EFA EEM").split()
START = "2010-01-01"
END = "2026-10-01"           # fixed end date, so the notebook always reproduces the same numbers
IS_END = "2017-12-31"        # in-sample: up to here; out-of-sample: afterwards

# --- the demonstration pair (Sweden vs Germany): see the note in section 1 ---
Y_TICKER, X_TICKER = "EWD", "EWG"
BASKET = ["EWD", "EWG", "EWN"]   # Sweden, Germany, Netherlands: basket for Johansen

# --- dynamic hedge ratio ---
ROLL_N = 252                 # rolling OLS window (1 year)
ALMA_N, ALMA_OFFSET, ALMA_SIGMA = 60, 0.85, 6

# --- signals (fixed, standard, not optimized) ---
ENTRY_Z, EXIT_Z, STOP_Z = 2.0, 0.5, 3.5
Z_WINDOW_MULT = 3            # z-score window = 3 x half-life (clipped)
Z_WINDOW_MIN, Z_WINDOW_MAX = 20, 120
TIME_STOP_MULT = 3           # close a trade open longer than 3 x half-life

# --- costs and metrics ---
COST = 0.0005                # 5 bps per side, on each leg
BARS_PER_YEAR = 252
SEED = 7

In [24]:
raw = yf.download(UNIVERSE, start=START, end=END, auto_adjust=True, progress=False)["Close"]
px = raw.dropna()                       # common history for every ETF
logpx = np.log(px)
IS = logpx.loc[:IS_END]
print(f"{px.shape[1]} ETFs, {len(px)} daily bars, {px.index[0].date()} -> {px.index[-1].date()}")
print(f"in-sample: {IS.index[0].date()} -> {IS.index[-1].date()} ({len(IS)} bars), out-of-sample: {len(px) - len(IS)} bars")

38 ETFs, 4211 daily bars, 2010-01-04 -> 2026-09-30
in-sample: 2010-01-04 -> 2017-12-29 (2013 bars), out-of-sample: 2198 bars


In [25]:
def add_oos(fig, row=None, col=None):
    fig.add_vrect(x0=IS_END, x1=px.index[-1], fillcolor="white", opacity=0.07, line_width=0, layer="below", row=row, col=col)

def plot_lines(series, title, ytitle=None, logy=False, height=500, oos=True, hlines=()):
    """Time-series helper: dict name -> Series. The out-of-sample period is shaded."""
    fig = go.Figure([go.Scatter(x=s.index, y=s, name=n, mode="lines", line_width=1.8) for n, s in series.items()])
    for y, color in hlines:
        fig.add_hline(y=y, line_dash="dash", line_color=color)
    if oos:
        add_oos(fig)
    fig.update_layout(title=title, yaxis_title=ytitle, height=height, hovermode="x unified")
    if logy:
        fig.update_yaxes(type="log")
    fig.show()

def sharpe(r):
    return r.mean() / r.std() * np.sqrt(BARS_PER_YEAR) if r.std() > 0 else np.nan

def max_dd(r):
    eq = (1 + r).cumprod()
    return (eq / eq.cummax() - 1).min()

def equity(r):
    return (1 + r).cumprod()

## 1. Data and the pair, by eye

Prices are adjusted closes from `yfinance`. All the statistics work on **log prices**, so that a hedge ratio has a percentage meaning.

A note on the choice of the pair. The pair below (`EWD` Sweden, `EWG` Germany) is the best-looking one found by the screening of section 7, which is run on the in-sample data only. This is a **demonstration pair**, selected among hundreds: the notebook will show, with numbers, why this makes it a much weaker candidate than it looks.

In [26]:
y, x = IS[Y_TICKER], IS[X_TICKER]
reb = px[[Y_TICKER, X_TICKER]] / px[[Y_TICKER, X_TICKER]].iloc[0] * 100
plot_lines({Y_TICKER: reb[Y_TICKER], X_TICKER: reb[X_TICKER]}, f"{Y_TICKER} and {X_TICKER}, rebased to 100 (shaded: out-of-sample)", logy=True)

The two ETFs move together for years: the correlation of their daily returns in-sample is 0.88 and that of their log prices 0.98. After the in-sample period (shaded area starts in 2018) the picture changes: `EWD` pulls away from `EWG` and the gap does not close. Keep this in mind: it is the whole story of the notebook.

## 2. Spurious correlation: prices vs returns

Correlation measures how much two assets move together **day by day**: it is computed on returns. Computed on prices it is almost always high and almost always useless, because any two trending series look correlated. The experiment: 1000 pairs of **completely independent** random walks, and the correlation of their prices against the correlation of their returns.

In [27]:
rng = np.random.default_rng(SEED)
n_obs, n_sim = len(IS), 1000
steps = rng.normal(0, 0.01, (n_sim, 2, n_obs))
walks = steps.cumsum(axis=2)
corr_prices = np.array([np.corrcoef(w[0], w[1])[0, 1] for w in walks])
corr_returns = np.array([np.corrcoef(s[0], s[1])[0, 1] for s in steps])

fig = go.Figure()
fig.add_trace(go.Histogram(x=corr_prices, name="correlation of prices", nbinsx=50, opacity=0.75))
fig.add_trace(go.Histogram(x=corr_returns, name="correlation of returns", nbinsx=50, opacity=0.75))
fig.update_layout(barmode="overlay", height=450, title="1000 pairs of independent random walks", xaxis_title="correlation")
fig.show()
print(f"prices:  mean |corr| = {np.abs(corr_prices).mean():.2f}, share with |corr| > 0.5 = {(np.abs(corr_prices) > 0.5).mean():.0%}")
print(f"returns: mean |corr| = {np.abs(corr_returns).mean():.2f}, share with |corr| > 0.5 = {(np.abs(corr_returns) > 0.5).mean():.0%}")

prices:  mean |corr| = 0.42, share with |corr| > 0.5 = 38%
returns: mean |corr| = 0.02, share with |corr| > 0.5 = 0%


Independent random walks have, on average, an absolute price correlation of 0.42, and 38% of the pairs have a correlation above 0.5 in absolute value, although they have nothing to do with each other. The correlation of their returns is around zero (mean absolute value 0.02, no pair above 0.5). Correlation on prices is a spurious correlation: any two trending series look related.

### Correlation ≠ cointegration

Two simulated cases (left to right in the figure):

- **Correlated, not cointegrated**: the daily returns are strongly correlated, but one asset has a slightly higher drift. The gap grows and never comes back.
- **Cointegrated, weakly correlated**: each series has its own random shocks, but both are pulled back toward a common level (the *leash*). The day-by-day correlation is low, the long-run link is tight.

Murray's drunk and her dog: if the dog is free, the two wander apart. On a leash each one still walks randomly, but the distance between them stays bounded. The leash is cointegration.

In [28]:
rng = np.random.default_rng(SEED)
n = 1500
# A) correlated returns (rho ~ 0.9), different drift: not cointegrated
common = rng.normal(0, 0.01, n)
a1 = (common + rng.normal(0, 0.0045, n)).cumsum()
a2 = (common + rng.normal(0, 0.0045, n) + 0.0006).cumsum()
# B) drunk and dog on a leash: both follow their own shocks, pulled toward each other
b1, b2 = np.zeros(n), np.zeros(n)
for t in range(1, n):
    gap = b1[t - 1] - b2[t - 1]
    b1[t] = b1[t - 1] - 0.08 * gap + rng.normal(0, 0.01)
    b2[t] = b2[t - 1] + 0.08 * gap + rng.normal(0, 0.01)
# C) the same two walks without the leash
c1, c2 = rng.normal(0, 0.01, n).cumsum(), rng.normal(0, 0.01, n).cumsum()

cases = {"A) correlated returns, no leash": (a1, a2), "B) on a leash (cointegrated)": (b1, b2), "C) free dog (no leash)": (c1, c2)}
fig = make_subplots(rows=2, cols=3, subplot_titles=list(cases) + ["gap A", "gap B", "gap C"], vertical_spacing=0.15)
for j, (name, (p, q)) in enumerate(cases.items(), 1):
    fig.add_trace(go.Scatter(y=p, name="asset 1", line_color="#00cc96", showlegend=(j == 1)), row=1, col=j)
    fig.add_trace(go.Scatter(y=q, name="asset 2", line_color="#ef553b", showlegend=(j == 1)), row=1, col=j)
    fig.add_trace(go.Scatter(y=p - q, name="gap", line_color="#ab63fa", showlegend=False), row=2, col=j)
fig.update_layout(height=600, title="Correlation and cointegration are different things")
fig.show()

rows = []
for name, (p, q) in cases.items():
    rows.append({"case": name, "corr of returns": np.corrcoef(np.diff(p), np.diff(q))[0, 1],
                 "corr of prices": np.corrcoef(p, q)[0, 1], "coint p-value": coint(p, q)[1]})
display(pd.DataFrame(rows).set_index("case").round(3))

,corr of returns,corr of prices,coint p-value
case,,,
"A) correlated returns, no leash",0.826,0.141,0.348
B) on a leash (cointegrated),-0.036,0.982,0.000
C) free dog (no leash),0.001,0.759,0.900


Case A has a return correlation of 0.83, yet the gap drifts away forever (cointegration p-value 0.35: not cointegrated). Case B has essentially **zero** correlation of returns (−0.04) but a price correlation of 0.98 and a cointegration p-value of 0.000: the leash is there. Case C (the free dog) has a high price correlation (0.76) and no cointegration (p = 0.90). Correlation of returns tells us nothing about whether the gap closes.

## 3. Stationarity and the ADF test

A **stationary** series has a stable mean and variance and tends to come back to its mean: this is the behaviour we want from a spread. Prices are usually **I(1)**: they behave like a random walk, but their differences (returns) are stationary, **I(0)**. Two I(1) prices are cointegrated if some combination of them, the spread, is I(0).

The Augmented Dickey-Fuller test regresses today's change on yesterday's level (plus lagged changes). If the series mean-reverts, the coefficient γ is negative; for a random walk it is zero. **H0: γ = 0, unit root, not stationary**: we want to *reject* it (statistic very negative, p-value < 0.05).

Intuition first: the scatter of Δy against y(t−1) for a random walk and for a mean-reverting (Ornstein-Uhlenbeck) series.

In [29]:
rng = np.random.default_rng(8)
n = 2000
rw = rng.normal(0, 1, n).cumsum()
ou = np.zeros(n)
for t in range(1, n):
    ou[t] = ou[t - 1] - 0.05 * ou[t - 1] + rng.normal(0, 1)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Random walk (unit root)", "Mean reverting (OU)"), shared_yaxes=True)
for j, s in enumerate([rw, ou], 1):
    lvl, dlt = s[:-1], np.diff(s)
    slope, icpt = np.polyfit(lvl, dlt, 1)
    fig.add_trace(go.Scatter(x=lvl, y=dlt, mode="markers", marker=dict(size=3, opacity=0.5), showlegend=False), row=1, col=j)
    xs = np.array([lvl.min(), lvl.max()])
    fig.add_trace(go.Scatter(x=xs, y=icpt + slope * xs, mode="lines", line=dict(color="orange", width=3),
                             name=f"slope γ = {slope:+.3f}"), row=1, col=j)
    fig.update_xaxes(title_text="y(t-1)", row=1, col=j)
fig.update_yaxes(title_text="Δy(t)", row=1, col=1)
fig.update_layout(height=450, title="ADF intuition: does yesterday's level predict today's change?")
fig.show()
print(f"ADF p-value  random walk: {adfuller(rw)[1]:.3f}   OU: {adfuller(ou)[1]:.4f}")

ADF p-value  random walk: 0.820   OU: 0.0000


Now the same three views on real data: a price (non-stationary), its returns (stationary) and a spread (stationary by construction in this demo, built in the next section). Then the ADF test on the log prices and on the log returns of **every** ETF of the universe, in-sample.

In [30]:
rets = IS.diff().dropna()
adf_p = pd.DataFrame({"prices": IS.apply(lambda s: adfuller(s, autolag="AIC")[1]),
                      "returns": rets.apply(lambda s: adfuller(s, autolag="AIC")[1])})
fig = go.Figure([go.Histogram(x=adf_p[c], name=f"ADF on {c}", xbins=dict(start=0, end=1, size=0.05), opacity=0.8) for c in adf_p])
fig.add_vline(x=0.05, line_dash="dash", line_color="white")
fig.update_layout(barmode="overlay", height=400, title="ADF p-values across the 38 ETFs (in-sample)", xaxis_title="p-value")
fig.show()
print(f"stationary (p < 0.05) on log prices:  {(adf_p.prices < 0.05).sum()} / {len(adf_p)}")
print(f"stationary (p < 0.05) on log returns: {(adf_p.returns < 0.05).sum()} / {len(adf_p)}")

stationary (p < 0.05) on log prices:  1 / 38
stationary (p < 0.05) on log returns: 38 / 38


On log prices the test almost never rejects the unit root: only 1 ETF of 38 has p < 0.05 (`EZA`, p = 0.016, which with 38 tests is what chance alone can produce). On log returns it rejects for all 38. This is the I(1) / I(0) structure we expect: prices are random-walk-like, returns are stationary.

## 4. Hedge ratio and spread

For two log prices, a linear regression gives $y_t = \alpha + \beta x_t + \varepsilon_t$: the slope β is the **hedge ratio** and the residual is the **spread** $s_t = y_t - \beta x_t$ (the intercept only shifts the level).

The trap: **OLS is not symmetric.** It minimizes the errors on the dependent variable only, so regressing *y on x* and regressing *x on y* (then inverting the slope) give different hedge ratios. **Total Least Squares** (orthogonal regression, the first principal component of the two series) treats both assets in the same way and falls between them.

In [31]:
def ols_beta(y, x):
    return sm.OLS(y, sm.add_constant(x)).fit().params.iloc[1]

def tls_beta(y, x):
    pts = np.c_[x - x.mean(), y - y.mean()]
    v = np.linalg.svd(pts, full_matrices=False)[2][0]     # first principal direction
    return v[1] / v[0]

beta_yx = ols_beta(y, x)                 # y on x
beta_xy = 1 / ols_beta(x, y)             # x on y, inverted
beta_tls = tls_beta(y, x)
print(f"OLS  y on x:          beta = {beta_yx:.3f}")
print(f"OLS  x on y (inv.):   beta = {beta_xy:.3f}")
print(f"TLS (orthogonal):     beta = {beta_tls:.3f}")

xs = np.array([x.min(), x.max()])
fig = go.Figure(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=4, opacity=0.5, color=np.arange(len(x)), colorscale="Viridis",
                                                                 colorbar=dict(title="time")), showlegend=False))
for name, b in [("OLS y on x", beta_yx), ("OLS x on y (inverted)", beta_xy), ("TLS", beta_tls)]:
    a = y.mean() - b * x.mean()
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", name=f"{name}: β = {b:.2f}", line_width=3))
fig.update_layout(height=550, title="Three hedge ratios for the same pair (log prices, in-sample)",
                  xaxis_title=f"log {X_TICKER}", yaxis_title=f"log {Y_TICKER}")
fig.show()

OLS  y on x:          beta = 0.912
OLS  x on y (inv.):   beta = 0.953
TLS (orthogonal):     beta = 0.931


The three hedge ratios are close but not equal: 0.91 (y on x), 0.95 (x on y, inverted) and 0.93 (TLS, in between). The difference is small here because the two ETFs are very strongly linked in-sample (the more perfect the fit, the smaller the asymmetry), but it grows when the fit is looser, and it can change the outcome of the cointegration test. That is why we test in both directions.

From here the **static** hedge ratio is the OLS of `y` on `x` estimated in-sample, and the spread is $s_t = y_t - \beta x_t$. The figure shows the spread in-sample and what happens to it afterwards, with the same frozen β.

In [32]:
BETA_STATIC = beta_yx
spread_static = logpx[Y_TICKER] - BETA_STATIC * logpx[X_TICKER]
spread_static -= spread_static.loc[:IS_END].mean()
plot_lines({"spread (static OLS beta, in-sample mean removed)": spread_static},
           f"{Y_TICKER} - {BETA_STATIC:.2f} x {X_TICKER}: in-sample vs out-of-sample", hlines=[(0, "white")])

## 5. Engle-Granger and the CADF test: mind the critical values

**Engle-Granger** is a two-step method: (1) OLS to estimate β and build the spread, (2) ADF test on the residuals. The second step is often called **CADF** (cointegrated ADF).

There is a technical detail that separates a correct test from a wrong one. β was estimated on the same data precisely to make the spread as stable as possible, so the regression has already done half of the work in favour of stationarity. The standard ADF critical values (about −2.86 at 5% with a constant) are therefore **too lenient**: for two variables we need **MacKinnon's** cointegration critical values (about −3.34). `statsmodels.tsa.stattools.coint` does everything: regression, test and correct critical values. Since the result depends on the order of the variables, we run it in both directions.

In [33]:
def eg_test(y, x):
    """Engle-Granger in both directions. Returns the table with the CADF (coint) and the naive ADF on the residuals."""
    rows = {}
    for name, (a, b) in {f"{y.name} on {x.name}": (y, x), f"{x.name} on {y.name}": (x, y)}.items():
        stat, p, crit = coint(a, b)
        res = sm.OLS(a, sm.add_constant(b)).fit().resid
        naive_stat, naive_p = adfuller(res, autolag="AIC")[:2]
        rows[name] = {"CADF stat": stat, "CADF p-value": p, "crit 5% (MacKinnon)": crit[1],
                      "naive ADF stat": naive_stat, "naive ADF p-value": naive_p}
    return pd.DataFrame(rows).T

eg = eg_test(y, x)
display(eg.round(4))

,CADF stat,CADF p-value,crit 5% (MacKinnon),naive ADF stat,naive ADF p-value
EWD on EWG,-5.3360,0.0000,-3.3392,-5.3393,0.0
EWG on EWD,-5.0776,0.0001,-3.3392,-5.0797,0.0


In [34]:
# a classic pair that is *not* cointegrated in-sample: Australia vs Canada
display(eg_test(IS["EWA"].rename("EWA"), IS["EWC"].rename("EWC")).round(4))

,CADF stat,CADF p-value,crit 5% (MacKinnon),naive ADF stat,naive ADF p-value
EWA on EWC,-2.4497,0.3017,-3.3392,-2.4490,0.1284
EWC on EWA,-2.7682,0.1760,-3.3392,-2.7674,0.0631


How much does the wrong critical value matter? On the 703 pairs of the universe (in-sample) we compare the p-value of the **naive** ADF on the OLS residuals with the correct Engle-Granger p-value from `coint`, both in the best direction.

In [35]:
def both_pvalues(a, b):
    naive, correct = [], []
    for u, v in ((a, b), (b, a)):
        naive.append(adfuller(sm.OLS(u, sm.add_constant(v)).fit().resid, autolag="AIC")[1])
        correct.append(coint(u, v)[1])
    return min(naive), min(correct)

cols = list(IS.columns)
pp = Parallel(n_jobs=-1)(delayed(both_pvalues)(IS[a].to_numpy(), IS[b].to_numpy()) for a, b in combinations(cols, 2))
cmp_ = pd.DataFrame(pp, columns=["naive ADF p", "CADF p (coint)"], index=[f"{a}/{b}" for a, b in combinations(cols, 2)])
print(f"pairs with p < 0.05:  naive ADF on residuals = {(cmp_['naive ADF p'] < 0.05).sum()},  correct Engle-Granger = {(cmp_['CADF p (coint)'] < 0.05).sum()}  (of {len(cmp_)})")
wrong = cmp_[(cmp_["naive ADF p"] < 0.05) & (cmp_["CADF p (coint)"] >= 0.05)]
print(f"'cointegrated' for the naive test but not for the correct one: {len(wrong)} pairs")
display(wrong.sort_values("naive ADF p").head(5).round(3))

pairs with p < 0.05:  naive ADF on residuals = 185,  correct Engle-Granger = 69  (of 703)
'cointegrated' for the naive test but not for the correct one: 116 pairs


,naive ADF p,CADF p (coint)
EWJ/IWM,0.013,0.050
EWA/XLE,0.013,0.050
EWJ/EWT,0.014,0.050
EWG/EWL,0.014,0.050
EWA/EWY,0.014,0.051


In [36]:
# the two critical values on the statistic axis, with the pair's best statistic
best = eg["CADF stat"].min()
crit_mk, crit_adf = eg["crit 5% (MacKinnon)"].iloc[0], adfuller(np.random.default_rng(0).normal(size=500))[4]["5%"]
fig = go.Figure()
fig.add_vline(x=crit_adf, line_color="orange", line_width=3, annotation_text=f"standard ADF 5%: {crit_adf:.2f}", annotation_position="top left")
fig.add_vline(x=crit_mk, line_color="red", line_width=3, annotation_text=f"Engle-Granger 5%: {crit_mk:.2f}", annotation_position="top right")
fig.add_trace(go.Scatter(x=[best], y=[0], mode="markers+text", marker=dict(size=16, color="#00cc96"),
                         text=[f"{Y_TICKER}/{X_TICKER}: {best:.2f}"], textposition="bottom center", showlegend=False))
fig.add_trace(go.Scatter(x=[-3.0], y=[0], mode="markers+text", marker=dict(size=16, color="white", symbol="diamond"),
                         text=["example: -3.0<br>passes ADF, fails cointegration"], textposition="top center", showlegend=False))
fig.update_xaxes(range=[-5, -1.5], title="test statistic (more negative = more stationary)")
fig.update_yaxes(visible=False, range=[-1, 1])
fig.update_layout(height=300, title="Where the statistic lands decides what we conclude")
fig.show()

In-sample, `EWD`/`EWG` is strongly cointegrated: the CADF statistic is −5.34 (and −5.08 in the other direction) against a 5% critical value of −3.34. `EWA`/`EWC`, one of the most quoted pairs, is **not** cointegrated in this sample (CADF p-values 0.30 and 0.18). A pair that is famous is not necessarily a pair that works.

The naive ADF on the residuals gives almost the same *statistic* as `coint`; the difference is in the critical values and p-values. Over the 703 pairs, the naive test finds 185 "cointegrated" pairs, the correct one 69: **116 pairs are false positives of the wrong critical value**, many of them just under the threshold (naive p around 0.013, correct p around 0.05).

## 6. The Johansen test on a 3-asset basket

Engle-Granger handles only two assets, depends on the order of the variables and finds at most one relationship. **Johansen** (1988) models all the assets together with a VECM:

$$\Delta \mathbf{y}_t = \Pi\, \mathbf{y}_{t-1} + \sum_i \Gamma_i \Delta \mathbf{y}_{t-i} + \varepsilon_t$$

The rank **r** of Π is the number of independent cointegrating relationships. The test is sequential: first H0: r = 0, if rejected then r ≤ 1, and so on; the **Trace** and **Max-Eigenvalue** statistics are compared with their critical values. The **eigenvectors** are the hedge ratios of the basket.

In [37]:
jo = coint_johansen(IS[BASKET], det_order=0, k_ar_diff=1)
n_b = len(BASKET)
tab = pd.DataFrame({"H0": [f"r <= {i}" for i in range(n_b)],
                    "trace stat": jo.lr1, "trace crit 95%": jo.cvt[:, 1],
                    "max-eig stat": jo.lr2, "max-eig crit 95%": jo.cvm[:, 1]}).set_index("H0")
tab["reject (trace)"] = tab["trace stat"] > tab["trace crit 95%"]
tab["reject (max-eig)"] = tab["max-eig stat"] > tab["max-eig crit 95%"]
display(tab.round(2))

rank = int(np.argmax(~tab["reject (trace)"].to_numpy())) if (~tab["reject (trace)"]).any() else n_b
print(f"estimated rank (trace): r = {rank}")

evec = pd.DataFrame(jo.evec, index=BASKET)
w = evec.iloc[:, 0] / evec.iloc[0, 0]          # first eigenvector, normalised on the first asset
display(w.rename("weight (1st eigenvector)").to_frame().T.round(3))

,trace stat,trace crit 95%,max-eig stat,max-eig crit 95%,reject (trace),reject (max-eig)
H0,,,,,,
r <= 0,52.20,29.80,41.51,21.13,True,True
r <= 1,10.69,15.49,10.51,14.26,False,False
r <= 2,0.19,3.84,0.19,3.84,False,False


estimated rank (trace): r = 1


,EWD,EWG,EWN
weight (1st eigenvector),1.0,-1.083,0.199


In [38]:
basket_spread = logpx[BASKET] @ w
basket_spread -= basket_spread.loc[:IS_END].mean()
plot_lines({"basket spread (1st eigenvector, in-sample mean removed)": basket_spread},
           "Basket spread from the Johansen eigenvector, weights estimated in-sample", hlines=[(0, "white")])
print(f"ADF p-value of the basket spread, in-sample:  {adfuller(basket_spread.loc[:IS_END])[1]:.4f}  (naive: weights estimated on the same data)")
print(f"ADF p-value of the basket spread, out-of-sample: {adfuller(basket_spread.loc[IS_END:].iloc[1:])[1]:.4f}  (frozen weights, genuinely out-of-sample)")

ADF p-value of the basket spread, in-sample:  0.0000  (naive: weights estimated on the same data)
ADF p-value of the basket spread, out-of-sample: 0.3778  (frozen weights, genuinely out-of-sample)


For the basket `EWD`, `EWG`, `EWN` the sequence of tests rejects r = 0 (trace 52.2 vs 29.8 critical), but **not** r ≤ 1 (10.7 vs 15.5): the estimated rank is **r = 1**, one stationary combination, and Trace and Max-Eigenvalue agree. The first eigenvector gives the basket weights: long 1 `EWD`, short 1.08 `EWG`, long 0.20 `EWN`. The ADF on the spread is not informative in-sample (the weights were estimated on the same data, p ≈ 0), and the frozen weights tell the real story: **out-of-sample the spread is no longer stationary** (ADF p = 0.38). Same message as the pair.

## 7. Universe screening, p-value heatmap and the multiple testing problem

How to find pairs: (1) an economically sensible universe (done: country and sector ETFs), (2) a cointegration test on each pair (Engle-Granger, **both directions**, keeping the smaller p-value), (3) half-life and stability filters, (4) out-of-sample validation. The output is a heatmap of p-values: each cell is a pair.

The catch: with 38 ETFs there are **703** pairs. A test at 5% is wrong, by definition, one time in twenty, so even with *no* cointegrated pair we would expect about 35 "discoveries". Two corrections: **Bonferroni** (threshold = 0.05 / number of tests, very strict) and **Benjamini-Hochberg** (controls the share of false positives among the selected pairs). The control experiment is a universe of 38 **independent random walks**, where by construction no pair is cointegrated.

In [39]:
def coint_pvalue(a, b):
    """Engle-Granger p-value in both directions, keeping the smaller one."""
    return min(coint(a, b)[1], coint(b, a)[1])

def pvalue_matrix(df):
    cols = list(df.columns)
    pairs = list(combinations(range(len(cols)), 2))
    pv = Parallel(n_jobs=-1)(delayed(coint_pvalue)(df.iloc[:, i].to_numpy(), df.iloc[:, j].to_numpy()) for i, j in pairs)
    mat = pd.DataFrame(np.nan, index=cols, columns=cols)
    for (i, j), p in zip(pairs, pv):
        mat.iloc[i, j] = mat.iloc[j, i] = p
    return mat

def pair_table(mat):
    iu = np.triu_indices(len(mat), 1)
    t = pd.DataFrame({"a": mat.index[iu[0]], "b": mat.columns[iu[1]], "p": mat.to_numpy()[iu]})
    t["p_bonf"] = multipletests(t.p, method="bonferroni")[1]
    t["p_bh"] = multipletests(t.p, method="fdr_bh")[1]
    return t.sort_values("p").reset_index(drop=True)

pm_real = pvalue_matrix(IS)
pairs_real = pair_table(pm_real)

rng = np.random.default_rng(SEED)
fake = pd.DataFrame(rng.normal(0, 0.01, (len(IS), IS.shape[1])).cumsum(axis=0), columns=IS.columns, index=IS.index)
pm_fake = pvalue_matrix(fake)
pairs_fake = pair_table(pm_fake)

summary = pd.DataFrame({
    "pairs tested": [len(pairs_real), len(pairs_fake)],
    "p < 0.05 (no correction)": [(pairs_real.p < 0.05).sum(), (pairs_fake.p < 0.05).sum()],
    "Bonferroni": [(pairs_real.p_bonf < 0.05).sum(), (pairs_fake.p_bonf < 0.05).sum()],
    "Benjamini-Hochberg": [(pairs_real.p_bh < 0.05).sum(), (pairs_fake.p_bh < 0.05).sum()]},
    index=["real ETFs (in-sample)", "independent random walks"])
display(summary)

,pairs tested,p < 0.05 (no correction),Bonferroni,Benjamini-Hochberg
real ETFs (in-sample),703,69,1,1
independent random walks,703,67,0,0


In [40]:
order = list(IS.columns)
fig = make_subplots(rows=1, cols=2, subplot_titles=("38 real ETFs, in-sample", "38 independent random walks"), horizontal_spacing=0.08)
scale = [[0, "#00cc96"], [0.25, "#00cc96"], [0.2500001, "#1f2630"], [1, "#444c5a"]]   # green = p < 0.05, grey otherwise
for j, mat in enumerate([pm_real, pm_fake], 1):
    fig.add_trace(go.Heatmap(z=mat.loc[order, order].to_numpy(), x=order, y=order, zmin=0, zmax=0.2, colorscale=scale,
                             showscale=False, hovertemplate="%{x} / %{y}: p = %{z:.3f}<extra></extra>"), row=1, col=j)
fig.update_layout(height=620, title="Cointegration p-values (green: p < 0.05). Random data produces green cells too.")
fig.update_yaxes(autorange="reversed")
fig.show()

Of 703 pairs, 69 have p < 0.05 on the real ETFs in-sample. On **independent random walks** there are 67. The real data is almost indistinguishable from pure noise at this threshold: most of those 69 "discoveries" are what chance produces (about 35 expected, and the correction exposes it). After the Bonferroni and the Benjamini-Hochberg corrections only **1 pair** survives on real data (`EWD`/`EWG`, adjusted p = 0.027) and none on random data. The green cells of the left heatmap are not so different from those of the right one.

### Do the selected pairs survive out-of-sample?

The real test of a screening: take the pairs selected in-sample (p < 0.05, no correction), and test cointegration again on the **out-of-sample** years. If the relationships were real, most of them should still be cointegrated.

In [41]:
OOS = logpx.loc[IS_END:].iloc[1:]
sel = pairs_real[pairs_real.p < 0.05].copy()
sel["p_oos"] = Parallel(n_jobs=-1)(delayed(coint_pvalue)(OOS[a].to_numpy(), OOS[b].to_numpy()) for a, b in zip(sel.a, sel.b))
print(f"pairs selected in-sample at 5%: {len(sel)}")
print(f"of which still cointegrated out-of-sample at 5%: {(sel.p_oos < 0.05).sum()}  ({(sel.p_oos < 0.05).mean():.1%})")
print(f"median out-of-sample p-value of the selected pairs: {sel.p_oos.median():.2f}")

fig = go.Figure(go.Scatter(x=sel.p, y=sel.p_oos, mode="markers", text=sel.a + " / " + sel.b, marker=dict(size=8)))
fig.add_hline(y=0.05, line_dash="dash", line_color="white")
fig.update_layout(height=450, title="Selected pairs: in-sample vs out-of-sample cointegration p-value",
                  xaxis_title="in-sample p-value", yaxis_title="out-of-sample p-value")
fig.show()
display(sel.head(8).round(4))

pairs selected in-sample at 5%: 69
of which still cointegrated out-of-sample at 5%: 1  (1.4%)
median out-of-sample p-value of the selected pairs: 0.37


,a,b,p,p_bonf,p_bh,p_oos
0,EWD,EWG,0.0000,0.0266,0.0266,0.4802
1,EWD,EWQ,0.0005,0.3219,0.1363,0.6431
2,EFA,EWD,0.0006,0.4500,0.1363,0.5791
3,EWS,EZA,0.0008,0.5454,0.1363,0.0731
4,EFA,EWG,0.0011,0.7758,0.1552,0.4621
5,EWH,XLB,0.0014,1.0000,0.1684,0.4672
6,EWJ,EWN,0.0025,1.0000,0.2552,0.1723
7,EWC,EZA,0.0034,1.0000,0.2967,0.2293


Out-of-sample, **1 of the 69** selected pairs (1.4%) is still cointegrated at 5%, and the median out-of-sample p-value is 0.37. Even the only pair that survives the multiple testing correction, `EWD`/`EWG`, has an out-of-sample p-value of 0.48. The relationship found in-sample did not persist. A correction reduces false positives, but it does not guarantee that a relationship is stable in time: that is why out-of-sample validation is the real test.

## 8. Half-life and Hurst exponent

A stationary spread is not enough: it must also be **fast**. The **half-life** comes from the regression of the spread's change on its previous level, $\Delta s_t = \lambda s_{t-1} + \mu + \varepsilon_t$ (the discrete version of the Ornstein-Uhlenbeck process). The time needed to recover half of the distance from the mean is $-\ln 2 / \lambda$. A spread that closes in a few days is tradable, one that takes two years is useless even if statistically perfect. The half-life also gives the strategy its time scale: z-score window and time stop.

The **Hurst exponent** is a cross-check: H < 0.5 mean reverting, H = 0.5 random walk, H > 0.5 trending. It is estimated from how the dispersion of the lagged differences grows with the lag. It is not a formal test.

In [42]:
def half_life(s):
    s = pd.Series(np.asarray(s))
    lam = sm.OLS(s.diff().dropna(), sm.add_constant(s.shift(1).dropna())).fit().params.iloc[1]
    return -np.log(2) / lam if lam < 0 else np.inf

def hurst(s, lags=range(2, 100)):
    s = np.asarray(s)
    tau = [np.std(s[l:] - s[:-l]) for l in lags]
    return np.polyfit(np.log(list(lags)), np.log(tau), 1)[0]

sp_is = spread_static.loc[:IS_END]
HL = half_life(sp_is)
print(f"{Y_TICKER}/{X_TICKER} in-sample spread:  half-life = {HL:.1f} days,  Hurst = {hurst(sp_is):.2f}")
rw_ref = np.random.default_rng(SEED).normal(0, 1, len(sp_is)).cumsum()
print(f"random walk (reference):        Hurst = {hurst(rw_ref):.2f}")
print(f"log price of {Y_TICKER}:                Hurst = {hurst(y):.2f}")

# an exponential decay: fraction of the initial distance left after k days
k = np.arange(0, int(4 * HL))
decay = 0.5 ** (k / HL)
fig = go.Figure(go.Scatter(x=k, y=decay, mode="lines", line_width=3))
fig.add_hline(y=0.5, line_dash="dash", line_color="white")
fig.add_vline(x=HL, line_dash="dash", line_color="orange", annotation_text=f"half-life = {HL:.0f} days")
fig.update_layout(height=400, title="Mean reversion: share of the initial distance still open", xaxis_title="days", yaxis_title="fraction of distance left")
fig.show()

EWD/EWG in-sample spread:  half-life = 25.0 days,  Hurst = 0.34
random walk (reference):        Hurst = 0.54
log price of EWD:                Hurst = 0.40


In [43]:
# half-life and Hurst of all the in-sample selected pairs (static OLS spread, both estimated in-sample)
stats_sel = []
for a, b in zip(sel.a, sel.b):
    beta = ols_beta(IS[a], IS[b])
    s = IS[a] - beta * IS[b]
    stats_sel.append((half_life(s), hurst(s)))
sel["half_life"], sel["hurst"] = zip(*stats_sel)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Half-life (days)", "Hurst exponent"))
fig.add_trace(go.Histogram(x=sel.half_life.clip(upper=400), nbinsx=30, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=sel.hurst, nbinsx=30, showlegend=False), row=1, col=2)
fig.add_vline(x=0.5, line_dash="dash", line_color="white", row=1, col=2)
fig.update_layout(height=400, title=f"The {len(sel)} pairs selected in-sample (half-life clipped at 400)")
fig.show()
print(sel[["half_life", "hurst"]].describe().round(2).T)

           count   mean     std    min    25%    50%    75%     max
half_life   69.0  97.12  126.38  25.00  44.53  52.76  71.87  697.19
hurst       69.0   0.40    0.04   0.29   0.36   0.40   0.44    0.49


The in-sample spread of `EWD`/`EWG` has a half-life of 25 days (a good scale for a daily strategy) and a Hurst exponent of 0.34, against 0.54 for a random walk. Careful with Hurst: the log price of `EWD` itself reads 0.40 (daily returns of these ETFs are slightly mean-reverting at short lags), so it is a cross-check and not a test; the spread is clearly lower. Across the 69 selected pairs the median half-life is 53 days (25th-75th percentile: 45-72, but with a tail up to 697 days) and the median Hurst is 0.40: most selected pairs are slow, `EWD`/`EWG`, with 25 days, is the fastest of the 69.

## 9. Dynamic hedge ratio: static vs rolling OLS vs ALMA

The relationship between two assets changes over time, so a β estimated years ago may be wrong today. Options:

- **Static OLS**: β estimated in-sample and frozen (what we have used so far).
- **Rolling OLS**: β re-estimated every bar on the last N bars. It adapts, but it is noisy: every jump in β is a rebalancing, which costs money.
- **Rolling OLS + ALMA filter**: the Arnaud Legoux moving average applied to the rolling β. Its weights are a Gaussian bell, $w_i = \exp\!\left(-\frac{(i-m)^2}{2s^2}\right)$ with $m = \text{offset}\,(N-1)$ and $s = N/\sigma$. The *offset* moves the bell toward the most recent bars (low lag), *sigma* sets how narrow it is (how aggressive the smoothing).

Two warnings, both enforced below. (1) The spread at time *t* uses the β estimated **up to t−1** (`shift(1)`), otherwise we introduce look-ahead. (2) A β that is too reactive chases prices and **absorbs real divergences**: the spread looks stationary only because β moves to make it so.

In [44]:
def alma_weights(n=ALMA_N, offset=ALMA_OFFSET, sigma=ALMA_SIGMA):
    m, s = offset * (n - 1), n / sigma
    w = np.exp(-((np.arange(n) - m) ** 2) / (2 * s ** 2))
    return w / w.sum()

def alma(series, n=ALMA_N, offset=ALMA_OFFSET, sigma=ALMA_SIGMA):
    w = alma_weights(n, offset, sigma)
    return series.rolling(n).apply(lambda v: v @ w, raw=True)

def rolling_beta(y, x, n=ROLL_N):
    return y.rolling(n).cov(x) / x.rolling(n).var()

def hedge_ratios(y, x, beta_static):
    """The three hedge ratios actually *usable* at bar t: estimated with data up to t-1."""
    roll = rolling_beta(y, x)
    return pd.DataFrame({"static": beta_static, "rolling OLS": roll, "rolling OLS + ALMA": alma(roll.dropna())}, index=y.index).shift(1)

betas = hedge_ratios(logpx[Y_TICKER], logpx[X_TICKER], BETA_STATIC)
plot_lines({c: betas[c].dropna() for c in betas}, f"Hedge ratio of {Y_TICKER} on {X_TICKER}: static vs rolling ({ROLL_N} bars) vs ALMA",
           ytitle="beta")

w = alma_weights()
fig = go.Figure(go.Bar(x=np.arange(-ALMA_N + 1, 1), y=w))
fig.update_layout(height=350, title=f"ALMA weights (N = {ALMA_N}, offset = {ALMA_OFFSET}, sigma = {ALMA_SIGMA}): the bell sits on the recent bars",
                  xaxis_title="bars ago (0 = most recent)", yaxis_title="weight")
fig.show()

In [45]:
rows = {}
for c in betas:
    b = betas[c].dropna()
    sp = (logpx[Y_TICKER] - b * logpx[X_TICKER]).dropna()
    sp_is_c, sp_oos_c = sp.loc[:IS_END], sp.loc[IS_END:].iloc[1:]
    rows[c] = {"mean |change of beta| per bar": b.diff().abs().mean(), "ADF p, spread out-of-sample": adfuller(sp_oos_c)[1],
               "half-life OOS (days)": half_life(sp_oos_c)}
display(pd.DataFrame(rows).T.round(4))

,mean |change of beta| per bar,"ADF p, spread out-of-sample",half-life OOS (days)
static,0.0000,0.3205,216.5467
rolling OLS,0.0033,0.1351,8971.2375
rolling OLS + ALMA,0.0028,0.0752,inf


Rolling OLS moves the hedge ratio by 0.0033 per bar on average, ALMA by 0.0028: the filter reduces the noise only a little with these parameters. The more interesting column is the out-of-sample ADF p-value of the spread: 0.32 with the static β, 0.14 with rolling OLS and 0.075 with ALMA. The dynamic β makes the spread look **more stationary**, but this is partly by construction: β follows the divergence (the half-life out-of-sample is 217 days with the static β and practically infinite with the others, which means no measurable mean reversion at all). A better-looking statistic is not evidence of a better strategy; the backtest will tell.

## 10. Z-score and trading signals

The spread is normalized with a rolling z-score, $z_t = (s_t - \bar s_{t,N}) / \sigma_{t,N}$, with the window tied to the half-life (`Z_WINDOW_MULT` × half-life, clipped). The rules, all fixed and standard (nothing is optimized):

| Rule | Value |
|---|---|
| Short the spread | z > +2 |
| Long the spread | z < −2 |
| Exit | z comes back within ±0.5 of zero |
| Stop | \|z\| > 3.5: the relationship may be broken |
| Time stop | trade open longer than 3 × half-life |

**Long spread** = long `y`, short β units of `x`. After a stop or a time stop we stay flat until |z| is back below the entry threshold, otherwise we would re-enter immediately. The signal is evaluated at the **close** of bar *t*; the position is held from bar *t+1* (`shift(1)`).

In [46]:
Z_WIN = int(np.clip(round(Z_WINDOW_MULT * HL), Z_WINDOW_MIN, Z_WINDOW_MAX))
print(f"half-life = {HL:.1f} days -> z-score window = {Z_WIN} bars, time stop = {int(TIME_STOP_MULT * HL)} bars")

def zscore(spread, win):
    return (spread - spread.rolling(win).mean()) / spread.rolling(win).std()

def signals(z, time_stop):
    """State machine on the z-score: +1 long spread, -1 short spread, 0 flat. Decided at the close of each bar."""
    zv = z.to_numpy()
    pos, held, blocked = 0, 0, False
    out = np.zeros(len(zv), dtype=int)
    reason = {}
    for t, v in enumerate(zv):
        if np.isnan(v):
            continue
        if blocked and abs(v) < ENTRY_Z:
            blocked = False
        if pos == 0:
            if not blocked and v < -ENTRY_Z:
                pos, held = 1, 0
            elif not blocked and v > ENTRY_Z:
                pos, held = -1, 0
        else:
            held += 1
            if abs(v) > STOP_Z:
                pos, blocked = 0, True; reason[t] = "stop"
            elif held > time_stop:
                pos, blocked = 0, True; reason[t] = "time"
            elif (pos == 1 and v >= -EXIT_Z) or (pos == -1 and v <= EXIT_Z):
                pos = 0; reason[t] = "target"
        out[t] = pos
    return pd.Series(out, index=z.index), reason

b_alma = betas["rolling OLS + ALMA"]
spread_alma = logpx[Y_TICKER] - b_alma * logpx[X_TICKER]
z = zscore(spread_alma, Z_WIN)
sig, why = signals(z, int(TIME_STOP_MULT * HL))
ex = pd.DataFrame({"date": z.index[list(why)], "reason": list(why.values())})
ex["period"] = np.where(ex.date <= pd.Timestamp(IS_END), "in-sample", "out-of-sample")
display(pd.crosstab(ex.period, ex.reason))

zz = z.dropna()
fig = go.Figure(go.Scatter(x=zz.index, y=zz, name="z-score", line_width=1.2))
for lvl, col, dash in [(ENTRY_Z, "orange", "dash"), (-ENTRY_Z, "orange", "dash"), (0, "white", "dot"), (STOP_Z, "red", "dash"), (-STOP_Z, "red", "dash")]:
    fig.add_hline(y=lvl, line_color=col, line_dash=dash)
chg = sig.diff().fillna(0)
for mask, name, sym, col in [((chg != 0) & (sig != 0), "entry", "triangle-up", "#00cc96"), ((chg != 0) & (sig == 0), "exit", "x", "white")]:
    d = zz.reindex(sig.index[mask]).dropna()
    fig.add_trace(go.Scatter(x=d.index, y=d, mode="markers", name=name, marker=dict(symbol=sym, size=8, color=col)))
add_oos(fig)
fig.update_layout(height=520, title=f"Z-score of the {Y_TICKER}/{X_TICKER} spread (ALMA hedge ratio), entries ±{ENTRY_Z}, stop ±{STOP_Z}")
fig.show()

half-life = 25.0 days -> z-score window = 75 bars, time stop = 74 bars


reason,stop,target,time
period,,,
in-sample,1,13,3
out-of-sample,1,8,10


The z-score window is 75 bars (3 × 25 days) and the time stop 74 bars. The chart shows the entries (triangles) and exits (crosses): the exit reasons change a lot between the two periods. In-sample 13 of 17 exits are at the target, 3 are time stops and 1 is a stop; out-of-sample only 8 of 19 are at the target and 10 are time stops (1 stop): the spread stays away from its mean for long periods, which is exactly what the time stop is there to catch.

## 11. Backtest with costs and the look-ahead trap

The strategy is market neutral by construction. With weights $w_y = +1$ and $w_x = -\beta$ (long spread), the return on the capital deployed on bar *t* is

$$r_t = \text{pos}_t \cdot \frac{r^y_t - \beta_t\, r^x_t}{1 + |\beta_t|} - \text{costs}$$

where the division by $1 + |\beta|$ is the gross exposure, so the figure is a return on the capital in use. Costs are `COST` on **each leg** and each side, applied to the traded notional. Borrowing costs of the short leg are not modelled (an extra drag on real stock pairs). Positions use $\beta_t$ estimated up to *t−1*, signals are computed at the close and executed from the next bar.

First the **look-ahead trap**: the "fake" version estimates β, the mean and the standard deviation of the spread on the **full sample** and then trades on that same sample. The "correct" version uses only the information available at each bar.

In [47]:
def backtest(ret_y, ret_x, beta, sig, cost=COST):
    """Daily returns on deployed capital. beta[t] was estimated with data up to t-1, sig[t] decided at the close of t."""
    pos = sig.shift(1).fillna(0)                       # signal at close t -> position from t+1
    gross = 1 + beta.abs()
    wy, wx = pos / gross, -pos * beta / gross          # leg weights as a share of deployed capital
    gross_ret = wy * ret_y + wx * ret_x
    turnover = wy.diff().abs().fillna(0) + wx.diff().abs().fillna(0)
    return (gross_ret - cost * turnover).dropna(), pos

def trade_table(ret, pos):
    """One row per trade: return on capital while the position is open."""
    tid = ((pos != 0) & ((pos != pos.shift(1)))).cumsum().where(pos != 0)
    g = ret.reindex(pos.index).fillna(0).groupby(tid)
    return pd.DataFrame({"ret": g.apply(lambda r: (1 + r).prod() - 1), "bars": g.size()})

ry, rx = px[Y_TICKER].pct_change(), px[X_TICKER].pct_change()

def run_pair(ly, lx, ry_, rx_, beta, win, hl):
    sp = ly - beta * lx
    sg, why = signals(zscore(sp, win), int(TIME_STOP_MULT * hl))
    r, pos = backtest(ry_, rx_, beta.fillna(0), sg)
    first = beta.dropna().index[0]
    return r.loc[first:], pos.loc[first:], why

# correct: IS beta frozen, rolling z-score, only past data. fake: full-sample beta, mean and std. Both judged on the out-of-sample years
r_ok, _, _ = run_pair(logpx[Y_TICKER], logpx[X_TICKER], ry, rx, betas["static"], Z_WIN, HL)
beta_full = ols_beta(logpx[Y_TICKER], logpx[X_TICKER])
s_full = logpx[Y_TICKER] - beta_full * logpx[X_TICKER]
z_full = (s_full - s_full.mean()) / s_full.std()
sig_full, _ = signals(z_full, 10 ** 9)
r_fake, _ = backtest(ry, rx, pd.Series(beta_full, index=px.index), sig_full)
r_ok, r_fake = (r.loc[IS_END:].iloc[1:] for r in (r_ok, r_fake))

plot_lines({"look-ahead (full-sample beta, mean, std)": equity(r_fake), "correct (in-sample beta, rolling z-score)": equity(r_ok)},
           f"{Y_TICKER}/{X_TICKER}, out-of-sample years: the look-ahead backtest looks better than the honest one (costs included)",
           ytitle="equity (start = 1)", oos=False)
print(f"Sharpe  look-ahead: {sharpe(r_fake):.2f}   correct: {sharpe(r_ok):.2f}")
print(f"Max DD  look-ahead: {max_dd(r_fake):.1%}   correct: {max_dd(r_ok):.1%}")

Sharpe  look-ahead: 0.43   correct: 0.26
Max DD  look-ahead: -7.3%   correct: -9.3%


On the out-of-sample years, the look-ahead version (β, mean and standard deviation of the whole sample) has a Sharpe ratio of 0.43 and a maximum drawdown of −7.3%; the honest version has 0.26 and −9.3%. It is a modest difference here (a short out-of-sample sample and a pair that does not trend that much), but it has a sign that never changes: knowing the future always helps, so a backtest with look-ahead can only be better than reality, never worse.

### The three hedge ratios, same rules, same costs

Static (in-sample OLS), rolling OLS and rolling OLS + ALMA, with exactly the same signal logic, parameters and costs. The comparison is made **out-of-sample only**: the rolling hedge ratios use only the past in every bar, so they have no in-sample fit to speak of (their in-sample years are just another period), while the static one is the only estimate that depends on the in-sample data. For the static version the in-sample Sharpe is shown too, to see how well it fits the data it was estimated on.

We look at Sharpe, drawdown, trades and **how much β moves**: the amount of rebalancing that the strategy would have to pay for in live trading (here only the turnover implied by changes in position size is costed, which is the reason why a noisy β is not penalised more).

In [48]:
results, reasons = {}, {}
for c in betas:
    r, pos, why = run_pair(logpx[Y_TICKER], logpx[X_TICKER], ry, rx, betas[c], Z_WIN, HL)
    results[c], reasons[c] = (r, pos), why

alma_start = results["rolling OLS + ALMA"][0].index[0]      # same start as the rolling versions
rows = {}
for c, (r, pos) in results.items():
    rr, pp = r.loc[IS_END:].iloc[1:], pos.loc[IS_END:].iloc[1:]
    tt = trade_table(rr, pp)
    rows[c] = {"Sharpe OOS": sharpe(rr), "Sharpe IS (static only)": sharpe(r.loc[alma_start:IS_END]) if c == "static" else np.nan,
               "max DD OOS": max_dd(rr), "trades OOS": len(tt), "win rate": (tt.ret > 0).mean(),
               "avg win": tt.ret[tt.ret > 0].mean(), "avg loss": tt.ret[tt.ret <= 0].mean(), "worst trade": tt.ret.min(),
               "time in market": (pp != 0).mean(), "beta changes (mean abs)": betas[c].diff().abs().mean()}
display(pd.DataFrame(rows).T.round(4))
plot_lines({c: equity(r.loc[IS_END:].iloc[1:]) for c, (r, _) in results.items()},
           f"{Y_TICKER}/{X_TICKER}: out-of-sample equity with the three hedge ratios (costs included)", ytitle="equity (start = 1)", oos=False)

,Sharpe OOS,Sharpe IS (static only),max DD OOS,trades OOS,win rate,avg win,avg loss,worst trade,time in market,beta changes (mean abs)
static,0.2566,0.9897,-0.0932,33.0,0.6970,0.0107,-0.0148,-0.0282,0.3869,0.0000
rolling OLS,0.4190,NaN,-0.0576,20.0,0.6500,0.0173,-0.0093,-0.0187,0.4547,0.0033
rolling OLS + ALMA,0.2723,NaN,-0.1259,19.0,0.4737,0.0337,-0.0186,-0.0601,0.5002,0.0028


Out-of-sample the three hedge ratios are close: Sharpe 0.26 for the static one (33 trades, win rate 70%, maximum drawdown −9.3%), 0.42 for rolling OLS (20 trades, 65%, −5.8%) and 0.27 for ALMA (19 trades, 47%, −12.6%). With 19 to 33 trades per version these differences are not significant: no hedge ratio is shown to be better. The profiles differ though: the static version has many small wins (about 1.1%) and larger losses (about 1.5%), ALMA the opposite (wins of 3.4% against losses of 1.9%, but only 47% of winners) and a worst trade of −6.0%. The dynamic versions also stay in the market longer (45-50% of the bars against 39%).

The in-sample column is only meaningful for the static hedge ratio, which is estimated on those same years: its Sharpe is 0.99 in-sample and 0.26 out-of-sample, the cost of fitting the data we then trade. The rolling versions have nothing to fit in-sample, which is why they are judged out-of-sample only.

## 12. Results: in-sample vs out-of-sample, and the relationship breakdown

The same system, with the same fixed parameters, on the **top pairs of the in-sample screening** (the ones with the lowest in-sample p-value, any sector), each one with its own half-life. With the **static** hedge ratio, estimated in-sample, we compare in-sample and out-of-sample: this shows how much of the performance is just a fit to the data used for the selection. The ALMA hedge ratio, which only uses the past, is evaluated out-of-sample only. Costs are included.

In [49]:
def pair_backtest(a, b, method):
    # one pair, with the 'static' (in-sample OLS) or 'rolling OLS + ALMA' hedge ratio, each pair with its own half-life
    ly, lx = logpx[a], logpx[b]
    beta0 = ols_beta(IS[a], IS[b])
    hl = half_life(IS[a] - beta0 * IS[b])
    win = int(np.clip(round(Z_WINDOW_MULT * hl), Z_WINDOW_MIN, Z_WINDOW_MAX))
    hr = hedge_ratios(ly, lx, beta0)
    start = hr["rolling OLS + ALMA"].dropna().index[0]          # same start for both methods
    r, pos, _ = run_pair(ly, lx, px[a].pct_change(), px[b].pct_change(), hr[method], win, hl)
    return r.loc[start:], pos.loc[start:], hl

TOP = 8
rows, curves = [], {}
for a, b in zip(sel.a[:TOP], sel.b[:TOP]):
    r_s, pos_s, hl = pair_backtest(a, b, "static")
    r_a, _, _ = pair_backtest(a, b, "rolling OLS + ALMA")
    rows.append({"pair": f"{a}/{b}", "half-life": hl,
                 "static: Sharpe IS": sharpe(r_s.loc[:IS_END]), "static: Sharpe OOS": sharpe(r_s.loc[IS_END:].iloc[1:]),
                 "static: trades": len(trade_table(r_s, pos_s)), "static: max DD": max_dd(r_s),
                 "ALMA: Sharpe OOS": sharpe(r_a.loc[IS_END:].iloc[1:])})
    curves[f"{a}/{b}"] = equity(r_s)
res = pd.DataFrame(rows).set_index("pair")
display(res.round(2))
plot_lines(curves, "Static hedge ratio: equity of the top in-sample pairs (shaded: out-of-sample)", ytitle="equity (start = 1)", height=550)

,half-life,static: Sharpe IS,static: Sharpe OOS,static: trades,static: max DD,ALMA: Sharpe OOS
pair,,,,,,
EWD/EWG,25.00,0.99,0.26,64,-0.09,0.27
EWD/EWQ,57.27,0.55,-0.13,41,-0.15,0.26
EFA/EWD,41.64,0.78,0.42,49,-0.06,0.18
EWS/EZA,35.91,1.08,0.45,61,-0.19,0.24
EFA/EWG,34.29,0.14,-0.31,53,-0.15,-0.04
EWH/XLB,40.28,0.78,-0.08,46,-0.19,-0.02
EWJ/EWN,33.03,-0.05,0.33,49,-0.11,-0.13
EWC/EZA,38.68,0.53,0.52,47,-0.09,0.81


Static hedge ratio, top 8 pairs of the in-sample screening: the in-sample Sharpe is positive for 7 of 8 (from 0.14 to 1.08, only `EWJ`/`EWN` is slightly negative at −0.05), the out-of-sample one is positive for 5 and negative for 3. For `EWD`/`EWG` the Sharpe goes from 0.99 to 0.26, for `EWD`/`EWQ` from 0.55 to −0.13, for `EWH`/`XLB` from 0.78 to −0.08. The ALMA hedge ratio, out-of-sample, is positive for 4 pairs and negative for 4. With 8 pairs and one split this is anecdotal, which is why the next cell uses all 69 pairs.

### Across all the selected pairs

The top pairs could be a lucky or unlucky sample, so the same backtest is run on **all** the pairs selected in-sample at 5%. For the static hedge ratio the in-sample and out-of-sample Sharpe ratios are compared; the ALMA version is summarized out-of-sample only.

In [50]:
all_rows = []
for a, b in zip(sel.a, sel.b):
    r_s, _, _ = pair_backtest(a, b, "static")
    r_a, _, _ = pair_backtest(a, b, "rolling OLS + ALMA")
    all_rows.append({"pair": f"{a}/{b}", "static IS": sharpe(r_s.loc[:IS_END]), "static OOS": sharpe(r_s.loc[IS_END:].iloc[1:]),
                     "ALMA OOS": sharpe(r_a.loc[IS_END:].iloc[1:])})
allres = pd.DataFrame(all_rows).set_index("pair").dropna()

fig = go.Figure(go.Scatter(x=allres["static IS"], y=allres["static OOS"], mode="markers", text=allres.index, marker=dict(size=9), name="static hedge ratio"))
lim = [min(allres.min().min(), -1), max(allres.max().max(), 1)]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(dash="dash", color="white"), name="OOS = IS"))
fig.add_hline(y=0, line_color="grey"); fig.add_vline(x=0, line_color="grey")
fig.update_layout(height=550, title=f"Static hedge ratio: Sharpe in-sample vs out-of-sample, {len(allres)} selected pairs", xaxis_title="Sharpe IS", yaxis_title="Sharpe OOS")
fig.show()
summ = pd.DataFrame({"static": {"mean Sharpe IS": allres["static IS"].mean(), "mean Sharpe OOS": allres["static OOS"].mean(),
                                "median Sharpe IS": allres["static IS"].median(), "median Sharpe OOS": allres["static OOS"].median(),
                                "positive IS": (allres["static IS"] > 0).sum(), "positive OOS": (allres["static OOS"] > 0).sum(),
                                "corr(IS, OOS)": allres[["static IS", "static OOS"]].corr().iloc[0, 1]},
                     "ALMA (OOS only)": {"mean Sharpe OOS": allres["ALMA OOS"].mean(), "median Sharpe OOS": allres["ALMA OOS"].median(),
                                         "positive OOS": (allres["ALMA OOS"] > 0).sum()}})
display(summ.round(2))
print(f"pairs: {len(allres)}")

,static,ALMA (OOS only)
mean Sharpe IS,0.55,NaN
mean Sharpe OOS,0.12,0.10
median Sharpe IS,0.55,NaN
median Sharpe OOS,0.12,0.11
positive IS,65.00,NaN
positive OOS,45.00,41.00
"corr(IS, OOS)",0.23,NaN


pairs: 69


On all 69 selected pairs the static hedge ratio has a mean Sharpe of 0.55 in-sample (positive for 65 of 69) and 0.12 out-of-sample (positive for 45 of 69): **about four fifths of the in-sample Sharpe disappears**. The correlation between the in-sample and out-of-sample Sharpe across pairs is only 0.23: being good in-sample says little about being good afterwards. Out-of-sample the ALMA hedge ratio is in line with the static one (mean Sharpe 0.10, median 0.11, positive for 41 of 69). An average Sharpe of about 0.1 is low: a portfolio of mostly weak strategies, with a screening that cannot tell the good from the lucky.

### The relationship breakdown

The biggest risk of the strategy. The rolling ADF p-value of the spread (window of 2 years, frozen in-sample β) shows how the evidence of stationarity evolves: when it climbs above 5% and stays there, the leash is gone, and a mean reversion strategy keeps adding to a losing position.

In [51]:
WIN_ADF = 504
sp_frozen = logpx[Y_TICKER] - BETA_STATIC * logpx[X_TICKER]
idx = sp_frozen.index[WIN_ADF::21]
roll_p = pd.Series([adfuller(sp_frozen.loc[:t].iloc[-WIN_ADF:], regression="c")[1] for t in idx], index=idx)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5], vertical_spacing=0.08,
                    subplot_titles=("Spread with the frozen in-sample beta", f"Rolling ADF p-value of the spread ({WIN_ADF} bars window)"))
fig.add_trace(go.Scatter(x=sp_frozen.index, y=sp_frozen - sp_frozen.loc[:IS_END].mean(), name="spread", line_width=1.2), row=1, col=1)
fig.add_trace(go.Scatter(x=roll_p.index, y=roll_p, name="ADF p-value", line_width=1.8), row=2, col=1)
fig.add_hline(y=0.05, line_dash="dash", line_color="orange", row=2, col=1)
for r in (1, 2):
    add_oos(fig, row=r, col=1)
fig.update_layout(height=650, hovermode="x unified", title=f"{Y_TICKER}/{X_TICKER}: how long does the leash hold?")
fig.show()
print(f"share of the time with rolling p < 0.05:  in-sample {(roll_p.loc[:IS_END] < 0.05).mean():.0%},  out-of-sample {(roll_p.loc[IS_END:] < 0.05).mean():.0%}")

share of the time with rolling p < 0.05:  in-sample 60%,  out-of-sample 3%


The rolling ADF p-value of the spread is below 5% for 60% of the time in-sample and for only 3% out-of-sample. The spread mean moves from 0 to an average of +0.16 out-of-sample in log terms (the in-sample standard deviation is 0.036): `EWD` has simply outperformed `EWG` since 2020 and the gap has never come back. This is the **relationship breakdown**: a mean reversion strategy buys the laggard again and again while the gap keeps widening, which is the typical profile of many small wins and rare large losses.

## Key takeaways

- **Correlation ≠ cointegration.** Correlation is about returns and the short term; pairs trading needs the leash, a stationary combination of prices.
- **Engle-Granger for pairs, Johansen for baskets**, with the correct (MacKinnon) critical values and the test run in both directions.
- **Half-life** decides whether a spread is tradable and gives the time scale of every parameter.
- A **dynamic hedge ratio** helps, but a β that is too reactive hides real divergences.
- Screening many pairs produces many false positives: **multiple testing correction and out-of-sample validation are not optional**.
- **The real risk is the relationship breaking down.**

All the backtests here are demonstrations: daily bars, ETFs, one in-sample/out-of-sample split, simplified costs (no borrowing cost, no execution risk on the two legs). This is educational material, not financial advice.